# Business Analytics 2, Week 5: Data Preparation in Excel (Python version)

Companion notebook for the Week 5 page of the BA2 course app. The page cleans the jam stand's first wholesale order export step by step in Excel; this notebook does the same steps in Python (pandas) so you can compare them.

**Run the cells in order**: the first loads `raw` and `customers`, and later cells build `df` from it. If you restart the runtime, run them again from the top.

The files are read from the course's GitHub repository. If that link ever fails, download `jam-wholesale-raw.csv` and `jam-customers.csv` from the Week 5 page, upload them with the Files panel on the left, and change `BASE` to `""`.

_Generated from the code blocks on the Week 5 page._

## 1. Load the two CSV files (the right way)

In [ ]:
import pandas as pd

BASE = "https://raw.githubusercontent.com/PA-WiT/BA2/main/public/data/"   # or upload the two CSVs and use their file names

# sep=";" (not a comma), UTF-8, and everything read as text on purpose: we convert each column deliberately.
raw = pd.read_csv(BASE + "jam-wholesale-raw.csv", sep=";", encoding="utf-8", dtype=str, keep_default_na=False)
customers = pd.read_csv(BASE + "jam-customers.csv", sep=";", dtype=str)
print(raw.shape)   # (26, 7)
raw.head()

## 2. Clean the text: spaces and capitals

In [ ]:
df = raw.copy()                                    # never edit the raw frame
for col in ["customer", "flavor"]:
    # split() with no argument also treats the non-breaking space as a space, then join puts single spaces back
    df[col] = df[col].str.split().str.join(" ")
df["flavor"] = df["flavor"].str.title()            # 'strawberry' -> 'Strawberry'
df["customer"].nunique(), df["flavor"].nunique()   # 13 customer spellings (7 real customers), 5 flavor spellings (4 real)

## 3. Give each column its real type

In [ ]:
df["order_id"] = df["order_id"].astype(int)
df["order_date"] = pd.to_datetime(df["order_date"], format="%d/%m/%Y")        # day first, said out loud
df["unit_price"] = df["unit_price"].str.replace(",", ".", regex=False).astype(float)   # '5,00' and '5.00' both -> 5.0
df["jars"] = df["jars"].astype(int)

## 4. Check the validation rules

In [ ]:
rules = {
    "jars not a whole number 1-500": ~df["jars"].between(1, 500),
    "price outside 3-8":             ~df["unit_price"].between(3, 8),
    "date outside June 2024":        ~df["order_date"].between("2024-06-01", "2024-06-30"),
    "flavor not in the list":        ~df["flavor"].isin(["Strawberry", "Blueberry", "Peach", "Lemon"]),
}
for name, failed in rules.items():
    print(name, df.loc[failed, "order_id"].tolist())
# jars [1014, 1024] | price [] | date [1017] | flavor [1020]

## 5. Fix what the rules caught

In [ ]:
df = df[df["customer"] != "TEST"]                                           # 1024: a till set-up row
df.loc[df["order_id"] == 1014, "jars"] = 24                                  # sign error, confirmed by the owner
df.loc[df["order_id"] == 1017, "order_date"] = pd.Timestamp("2024-06-21")    # typed 2025
df.loc[df["order_id"] == 1020, "flavor"] = "Peach"                           # 'Peech'

## 6. Find and remove duplicates

In [ ]:
print(raw.duplicated().sum())          # 1: on the RAW text only the exact copy (1011) matches
print(df.duplicated().sum())           # 2: after cleaning, the copy of 1007 matches too
print(df[df.duplicated(subset="order_id", keep=False)][["order_id", "customer", "jars"]])
df = df.drop_duplicates(subset="order_id", keep="first")   # same order_id and identical rows: true copies
len(df)                                # 23

## 7. Look for outliers (IQR and z-score)

In [ ]:
q1, q3 = df["jars"].quantile([0.25, 0.75])      # same as QUARTILE.INC in Excel
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(q1, q3, low, high)                         # 36.0 60.0 0.0 96.0
print(df[(df["jars"] < low) | (df["jars"] > high)][["order_id", "customer", "jars"]])   # 1018: 360 jars

z = (df["jars"] - df["jars"].mean()) / df["jars"].std()   # .std() is the sample standard deviation, like STDEV.S
print(z.round(2).max())                          # 4.44

df.loc[df["order_id"] == 1018, "jars"] = 36      # only after the owner confirmed the extra zero

## 8. Look up the customer (exact match)

In [ ]:
df["key"] = df["customer"].str.lower()                 # lookups ignore case, so compare lower case
customers["key"] = customers["customer_name"].str.lower()

joined = df.merge(customers[["key", "customer_name", "region"]], on="key", how="left")
print(joined["region"].isna().sum())                  # 4 orders found no customer
print(joined.loc[joined["region"].isna(), ["order_id", "customer"]])
print(len(df.merge(customers, on="key", how="inner")))   # 19: an inner join drops those 4 silently

## 9. Suggest fuzzy matches for the misses

In [ ]:
import difflib

canon = {n.lower(): n for n in customers["customer_name"]}
unmatched = joined.loc[joined["region"].isna(), "customer"].unique()
for name in unmatched:
    close = difflib.get_close_matches(name.lower(), list(canon), n=2, cutoff=0.6)
    print(name, "->", [(canon[c], round(difflib.SequenceMatcher(None, name.lower(), c).ratio(), 2)) for c in close])
# Cafe Rose             -> [('Café Rosé', 0.78)]
# Greengrocer Ltd.      -> [('Green Grocer Ltd', 0.94), ('Greenfield Grocer Ltd', 0.81)]
# Harbor Market         -> [('Harbour Market', 0.96)]
# Green Grocers Limited -> [('Green Grocer Ltd', 0.86), ('Greenfield Grocer Ltd', 0.76)]

## 10. Approve the matches and finish

In [ ]:
# After reading the scores, approve the mapping by hand and keep it as the log of this step
approved = {
    "Cafe Rose": "Café Rosé",
    "Greengrocer Ltd.": "Green Grocer Ltd",
    "Harbor Market": "Harbour Market",
    "Green Grocers Limited": "Green Grocer Ltd",
}
df["customer"] = df["customer"].map(lambda s: approved.get(s, canon.get(s.lower(), s)))

final = df.merge(customers[["customer_name", "region"]], left_on="customer", right_on="customer_name",
                 how="left", validate="m:1")          # m:1 fails loudly if the customer list has duplicates
final["revenue"] = final["jars"] * final["unit_price"]
print(len(final), final["jars"].sum(), final["revenue"].sum())   # 23 1104 5520.0
final.groupby("region")["revenue"].sum()                          # Central 1728, Coast 1698, North 2094